<a href="https://colab.research.google.com/github/krithi-ks/flyrankAI-ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krithi-ks/flyrankAI-ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ML-05 setup
# Connect to the March 2026 development partition.

%pip -q install duckdb huggingface_hub

import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. Add your READ Hugging Face token "
        "to Colab Secrets as HF_TOKEN."
    )

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

PERFORMANCE = (
    "read_parquet("
    "'hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet'"
    ")"
)

print("Warehouse connection ready.")
print("Development window: March 2026")

Warehouse connection ready.
Development window: March 2026


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature vector

For the Lane 2 ranking task, I will use five initial features from the March 2026 development window. The feature vector is built at the **client × content** level.

The five features are:

1. **GSC impressions** — total search impressions observed during March.
2. **GSC clicks** — total search clicks observed during March.
3. **Average search position** — average observed search position during March, excluding `0` because `0` means no position data.
4. **GA4 sessions** — total sessions observed during March when GA4 data is available.
5. **Observed days** — number of distinct days with performance observations during March.

The features are limited to information available in the development window and do not include the future outcome or label-derived trend fields.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-05 Section 1
# Build the five-feature vector.
# One row = one client × content item.

five_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        -- Feature 1: search exposure
        SUM(gsc_impressions) AS impressions_mar,

        -- Feature 2: search response
        SUM(gsc_clicks) AS clicks_mar,

        -- Feature 3: average search position
        AVG(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
                ELSE NULL
            END
        ) AS avg_position_mar,

        -- Feature 4: GA4 sessions
        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN ga4_sessions
                ELSE NULL
            END
        ) AS sessions_mar,

        -- Feature 5: observed data coverage
        COUNT(DISTINCT report_date) AS observed_days_mar

    FROM {PERFORMANCE}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print(f"Feature rows: {len(five_features):,}")
print("Feature columns:")
print(list(five_features.columns))

display(five_features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 331,437
Feature columns:
['client_hash_id', 'content_hash_id', 'impressions_mar', 'clicks_mar', 'avg_position_mar', 'sessions_mar', 'observed_days_mar']


,client_hash_id,content_hash_id,impressions_mar,clicks_mar,avg_position_mar,sessions_mar,observed_days_mar
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,4.888929,NaN,31
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,602.0,4.0,4.428747,NaN,31
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,810.0,1.0,4.866123,NaN,31
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,82.0,0.0,10.100347,NaN,31
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,1858.0,6.0,1.854929,NaN,31


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

| Feature             | Meaning                                       | Missing-value handling                                                 | Available when?                                                       |
| ------------------- | --------------------------------------------- | ---------------------------------------------------------------------- | --------------------------------------------------------------------- |
| `impressions_mar`   | Total GSC impressions during March            | `SUM` naturally ignores missing values; no artificial zero is inserted | Before the ranking decision, because it summarizes the feature window |
| `clicks_mar`        | Total GSC clicks during March                 | `SUM` naturally ignores missing values                                 | Before the ranking decision, because it summarizes the feature window |
| `avg_position_mar`  | Average GSC position during March             | `0` is converted to missing because `0` means no position data         | Before the ranking decision                                           |
| `sessions_mar`      | GA4 sessions during March                     | Only rows where `ga4_data_available IS TRUE` contribute                | Before the ranking decision when GA4 data is available                |
| `observed_days_mar` | Number of days with observed performance data | Count is based on observed dates                                       | Before the ranking decision                                           |

The two hash identifiers are retained only to identify the client-content unit and are not intended as predictive features. No categorical variables are used in this initial five-feature vector.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check missingness in the five features.

feature_missingness = five_features[
    [
        "impressions_mar",
        "clicks_mar",
        "avg_position_mar",
        "sessions_mar",
        "observed_days_mar"
    ]
].isna().mean().mul(100).round(2)

missingness_check = feature_missingness.reset_index()
missingness_check.columns = ["feature", "missing_percent"]

display(missingness_check)

,feature,missing_percent
0,impressions_mar,0.00
1,clicks_mar,0.00
2,avg_position_mar,47.11
3,sessions_mar,72.70
4,observed_days_mar,0.00


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage hunt

I will check the feature frame for fields that directly represent or are derived from the target outcome. In particular, `trend_pct`, `trend_direction`, and `is_declining_label` must not be used as features because the data documentation states that the declining label is derived from trend information.

I will also check that future-window fields are not included in the five-feature vector. The March development features must only use information from the March feature window; future outcome information belongs to evaluation, not to the input features.

A feature that contains information derived from the outcome can make a model appear much better than it really is. Therefore, any such field is removed from the honest feature set.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# ML-05 Section 3 — Leakage Hunt
# Check that known label-derived or future-looking fields are not used
# in the feature vector.

# Get the actual columns available in the March performance partition
available_columns = con.sql(f"""
    DESCRIBE SELECT *
    FROM {PERFORMANCE}
""").df()["column_name"].tolist()

print("Number of available columns:", len(available_columns))

# Fields that must not be used as features because they represent
# the outcome/trend or future information.
known_leakage_fields = [
    "trend_pct",
    "trend_direction",
    "is_declining_label",
    "decline_label",
    "future_trend",
    "future_clicks",
    "future_impressions",
    "future_sessions"
]

# Check which of these actually exist in the warehouse
found_leakage_fields = [
    col for col in known_leakage_fields
    if col in available_columns
]

print("Known leakage/label-derived fields found:")
print(found_leakage_fields)

# Check the actual feature vector
feature_columns = [
    "impressions_mar",
    "clicks_mar",
    "avg_position_mar",
    "sessions_mar",
    "content_age_days"
]

feature_leakage = [
    col for col in feature_columns
    if col in known_leakage_fields
]

print("\nLeakage fields used in feature vector:")
print(feature_leakage)

if len(feature_leakage) == 0:
    print("PASS: The five-feature vector contains no known label-derived fields.")
else:
    print("FAIL: Leakage detected:", feature_leakage)



Number of available columns: 31
Known leakage/label-derived fields found:
[]

Leakage fields used in feature vector:
[]
PASS: The five-feature vector contains no known label-derived fields.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*


* **`client_hash_id`** — excluded from model features because it is a pseudonymous identifier and should only be used for grouping, joining, and splitting.

* **`content_hash_id`** — excluded from model features because it is a pseudonymous content identifier and should not be used for the model to learn from.

* **Future-period performance fields** — excluded because clicks, impressions, sessions, or other measurements from a future window would not be available at the decision moment.

* **Label-derived trend fields such as `trend_pct` and `trend_direction`** — excluded because they describe or derive from the outcome and would cause target leakage if used as features.

* **`is_declining_label`** — excluded because it is a label rather than a feature and directly represents the outcome being predicted.

* **June 2026 final-month data** — excluded from development because it is the natural future outcome window and is reserved as a sealed test month.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-05 Section 4 — Confirm excluded fields are not in the feature vector

excluded_fields = [
    "trend_pct",
    "trend_direction",
    "is_declining_label",
    "future_trend",
    "future_clicks",
    "future_impressions",
    "future_sessions",
    "client_hash_id",
    "content_hash_id"
]

used_features = [
    "impressions_mar",
    "clicks_mar",
    "avg_position_mar",
    "sessions_mar",
    "content_age_days"
]

excluded_but_used = [
    field for field in excluded_fields
    if field in used_features
]

print("Excluded fields:")
for field in excluded_fields:
    print("-", field)

print("\nExcluded fields accidentally used as features:")
print(excluded_but_used)

if not excluded_but_used:
    print("\nPASS: No intentionally excluded fields are being used as model features.")
else:
    print("\nCHECK REQUIRED: An excluded field is present in the feature vector.")


Excluded fields:
- trend_pct
- trend_direction
- is_declining_label
- future_trend
- future_clicks
- future_impressions
- future_sessions
- client_hash_id
- content_hash_id

Excluded fields accidentally used as features:
[]

PASS: No intentionally excluded fields are being used as model features.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.